In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Data Exploration - Monorail

## Data Kit 01, 06, 27

In [ ]:
import pandas as pd
import os
import re

def load_Monorail(filepath):
    df = pd.read_csv(filepath)
    df = df[df['Non_Standard_Braking'] == 0]

    # Extract numeric part from filename and convert to integer
    match = re.search(r'kit(\d+)', os.path.basename(filepath))
    source = int(match.group(1)) if match else -1  # fallback to -1 if no match
    df['Source'] = source

    for col in df.select_dtypes(include='object'):
        try:
            df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
        except ValueError:
            continue
    return df


# List of file paths
filepaths = [
    'TestBrakefinal_data_kit01.csv',
    'TestBrakefinal_data_kit06.csv',
    'TestBrakefinal_data_kit27.csv'
]

# Process all files
dfs = [load_Monorail(fp) for fp in filepaths]

# Combine into one DataFrame
df_data = pd.concat(dfs, ignore_index=True)

print(df_data[['Source']].value_counts())
print(df_data.shape)

### Data kit01 - Fills missing NaN with median value

In [ ]:
# from sklearn.impute import SimpleImputer
# import pandas as pd

# # Assuming df_data is already defined as in your code
# imputer = SimpleImputer(strategy='median')

# # Apply imputer to numeric columns only
# df_data[df_data.select_dtypes(include='number').columns] = imputer.fit_transform(df_data.select_dtypes(include='number'))

# # df_filtered = df_data[(df_data['WV_MeanPressure'] >= 2) & (df_data['WV_MeanPressure'] <= 3)]
# # print(df_filtered.shape)
# # df_filtered.head()

## San Donato Data

In [ ]:
import pandas as pd
import numpy as np

df_reference = pd.read_csv('model.csv')
df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

# Add source column
df_reference['Source'] = '0'

# Create binary label
leakage_codes = ['C', 'D', 'E', 'F', 'G']
df_reference['LeakageLabel'] = np.where(
    df_reference['Malfunction'].isin(leakage_codes),
    'Combined leakage',
    'Healthy'
)

# Aggregate delay and efficiency columns
delay_eff_map = {
    'Total_timing_delay': ['Brake_timing_delay_exp', 'Release_timing_delay_exp'],
    'Total_energy_delay': ['Brake_energy_delay_exp', 'Release_energy_delay_exp'],
    'Total_power_delay': ['Brake_power_delay_exp', 'Release_power_delay_exp'],
    'Total_power_efficiency': ['Brake_power_efficiency_exp', 'Release_power_efficiency_exp'],
    'Total_energy_efficiency': ['Brake_energy_effiency_exp', 'Release_energy_efficiency_exp']
}

for new_col, sources in delay_eff_map.items():
    df_reference[new_col] = df_reference[sources[0]] + df_reference[sources[1]]

# Drop original columns
cols_to_drop = [col for pair in delay_eff_map.values() for col in pair]
df_reference.drop(columns=cols_to_drop, inplace=True)

# Rename columns
rename_map = {
    'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
    'Buildup_end_pressure_delay_exp': 'Buildup_end_pressure_delay',
    'Weight': 'WV_MeanPressure',
    'Brake_action': 'EmergencyBrake_action'
}
df_reference.rename(columns=rename_map, inplace=True)

### Feature Engineering 

Split labeled data to features and target 

In [ ]:
common_cols = df_reference.columns.intersection(df_data.columns)

Features = df_reference[common_cols]
Features = Features.drop(columns=['LeakageLabel','WV_MeanPressure','EmergencyBrake_action'])
Parameters = df_reference[['WV_MeanPressure','EmergencyBrake_action','Brake_mode','Frequency','Sensor']]
Target = df_reference['LeakageLabel']
Target_raw = df_reference['Malfunction']

print(Features.shape)
Features.head()

Fills missing with median value instead of removing since we have little data

In [ ]:
# Impute missing (with median)
imp = SimpleImputer(strategy="median")
Features = pd.DataFrame(imp.fit_transform(Features), columns=Features.columns, index=Features.index)
#Features = Features.dropna(axis=1)

# Drop columns that are all NaN or constant
Features = Features.loc[:, Features.notna().any()]  # drop all-NaN
const_mask = Features.nunique(dropna=True) <= 1
if const_mask.any():
    Features = Features.loc[:, ~const_mask]

Features.shape
Features.columns

### Exploration - Feature Importance

In [ ]:
# === Unified Feature Selection Pipeline: RF, MI, ANOVA
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE, RFECV
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold

Xsel = Features.copy()

# Keep only numeric columns (if any non-numeric slipped in)
num_cols = [c for c in Xsel.columns if np.issubdtype(Xsel[c].dtype, np.number)]
Xsel = Xsel[num_cols].copy()

# Example: drop features with variance below 1e-2 AFTER scaling (optional):
vt = VarianceThreshold(threshold=1e-2)
X_var = vt.fit_transform(Xsel)
kept_mask = vt.get_support()
kept_features = Xsel.columns[kept_mask]
Xsel = Xsel[kept_features]


# Some selectors need scaling
sc_std = StandardScaler()
sc_rob = RobustScaler()
X_std = pd.DataFrame(sc_std.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)
X_rob = pd.DataFrame(sc_rob.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)

# if y is string, change to 0/1
y_enc = pd.Series(Target).astype("category")
if y_enc.dtype.name == "category":
    y_enc = y_enc.cat.codes  # e.g., Leakage=1, Normal=0

# For stability on tiny datasets
cv = StratifiedKFold(n_splits=min(5, max(2, np.bincount(y_enc).min())), shuffle=True, random_state=42)

# Helper to convert scores to ranks (lower rank = better)
def to_rank(series, higher_is_better=True):
    s = series.copy()
    if not higher_is_better:
        s = -s
    # rank 1 = best
    return s.rank(ascending=False, method="average")

# ---------- 1) RandomForest importance ----------
# Measures a feature's utility in improving the model's prediction accuracy (e.g., mean decrease in impurity).
# Captures feature interactions naturally; highly effective.
# REDUCED trees for small dataset stability
rf = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=3, 
                            random_state=42, class_weight="balanced")
rf.fit(Xsel, y_enc)
rf_imp = pd.Series(rf.feature_importances_, index=Xsel.columns, name="RF_Importance")
rf_rank = to_rank(rf_imp, higher_is_better=True).rename("RF_Rank")

# ---------- 2) Mutual Information ----------
# Measures statistical dependency (information gain) between a feature and the target.
# Captures non-linear relationships. Evaluates each feature independently; ignores feature interactions.
mi = mutual_info_classif(Xsel, y_enc, random_state=42, discrete_features=False, n_neighbors=3)
mi_score = pd.Series(mi, index=Xsel.columns, name="MI_Score")
mi_rank = to_rank(mi_score, higher_is_better=True).rename("MI_Rank")

# ---------- 3) ANOVA F-test ----------
# (works best if roughly Gaussian/scaled; we used imputed data)
# Measures linear correlation between a feature and the target by comparing variance between groups to variance within groups.
# Assumes linear relationship and Gaussian distribution; ignores feature interactions.
F_vals, p_vals = f_classif(Xsel, y_enc)
f_score = pd.Series(F_vals, index=Xsel.columns, name="ANOVA_F")
f_rank = to_rank(f_score, higher_is_better=True).rename("ANOVA_Rank")

# ---------- Combine all rankings with OPTIMIZED WEIGHTS for 60 samples ----------
rank_table = pd.concat([rf_rank, mi_rank, f_rank,
                        rf_imp, mi_score, f_score], axis=1)

# WEIGHTED OverallRank for small datasets (60 samples)
# MI: 0.50 (highest weight - most reliable for small data)
# ANOVA: 0.30 (second - stable if linear relationships exist)
# RF: 0.20 (lowest - prone to overfitting with 60 samples)
weights = {
    "MI_Rank": 0.6,
    "ANOVA_Rank": 1,
    "RF_Rank": 0.20
}

rank_table["OverallRank"] = (
    rank_table["MI_Rank"] * weights["MI_Rank"] +
    rank_table["ANOVA_Rank"] * weights["ANOVA_Rank"] +
    rank_table["RF_Rank"] * weights["RF_Rank"]
)

# Sort and display top-N
N = 7
rank_table_sorted = rank_table.sort_values("OverallRank").head(N)
print("=== Top features by Weighted OverallRank (lower = better) ===")
print(f"Weights: MI={weights['MI_Rank']}, ANOVA={weights['ANOVA_Rank']}, RF={weights['RF_Rank']}")
display(rank_table_sorted)

plt.figure(figsize=(8, max(4, 0.35*N)))
rank_table_sorted.sort_values("OverallRank")["OverallRank"].plot(kind="barh")
plt.gca().invert_yaxis()
plt.title(f"Top {N} Features by Weighted Rank (More on Anova)")
plt.xlabel("Rank (lower is better)")
plt.tight_layout()
plt.show()

topN_features = rank_table_sorted.index.tolist()
print("\nTopN feature list:", topN_features)

Features_reduced = Features[topN_features]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.heatmap(Features_reduced.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================
# CORRELATION PRUNING USING OVERALL RANK (lower = better)
# =============================================================

corr = Features_reduced.corr().abs()

# mask for upper triangle
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

threshold = 0.90
to_drop = []
clusters = []  # for reporting

for col in upper.columns:
    # features correlated with "col"
    correlated = list(upper.index[upper[col] > threshold])

    if len(correlated) > 0:
        group = [col] + correlated
        group = list(set(group))

        # record the group
        clusters.append(group)

        # choose the best feature based on OverallRank
        best_feature = (
            rank_table.loc[group]["OverallRank"]
            .sort_values()
            .index[0]
        )

        # others must be dropped
        drop_these = [f for f in group if f != best_feature]
        to_drop.extend(drop_these)

# make unique
to_drop = list(set(to_drop))

# =============================================================
# APPLY DROPPING
# =============================================================
final_features = [f for f in Features_reduced.columns if f not in to_drop]
X_final = Features_reduced[final_features]


# =============================================================
# PRINT REPORT
# =============================================================
print("=== Correlation Groups (|rho| > 0.90) ===")
for g in clusters:
    print(f"Group: {g}")
    best = rank_table.loc[g]["OverallRank"].sort_values().index[0]
    print(f"→ Keeping: {best}")
    print()

print("\n=== Features DROPPED due to high correlation ===")
print(to_drop)

print("\n=== FINAL FEATURE LIST ===")
print(final_features)


# =============================================================
# PLOT FINAL FEATURES
# =============================================================
plt.figure(figsize=(8, max(3, 0.35 * len(final_features))))
plt.barh(final_features, [1]*len(final_features))
plt.title("Final Selected Features After Correlation Pruning")
plt.xlabel("Kept (1 = yes)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
Feature_use = Features_reduced[final_features]
sns.heatmap(Feature_use.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
def box_target_plotter(data, target):
    for col in data.select_dtypes("number"):
        sns.boxplot(data=data, x=target, y=col)
        plt.show()

box_target_plotter(Feature_use, Target)

## Combine Reference data with San Donato data

In [ ]:
# make sure the Source columns are of the same type
df_reference['Source'] = df_reference['Source'].astype(int)
df_data['Source'] = df_data['Source'].astype(int)

# Combine common columns
common_cols = df_reference.columns.intersection(df_data.columns).tolist()

# Add DataSource column to each DataFrame
df_reference_subset = df_reference[common_cols].copy()
df_reference_subset['DataSource'] = 0

df_data_subset = df_data[common_cols].copy()
df_data_subset['DataSource'] = 1

# Concatenate the two DataFrames
df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

# Final label encoding
df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

# Convert ' sec' strings to float
for col in df_combined.select_dtypes(include='object'):
    try:
        df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
    except ValueError:
        continue

df_base = df_combined.copy()

Target_combined = df_base['label']
print(df_base.shape)
df_base.head()

Intersect of Data Reference with San Donato Data to compare features

In [ ]:
df_filtered = df_combined[(df_combined['WV_MeanPressure'] >= 2) & (df_combined['WV_MeanPressure'] <= 3)]
Target_data = df_combined['label']

df_filtered.head()

### Combined Leakage Boxchart of Features 

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ---- inputs ----
# df: your combined dataframe (already built above)
# Features_reduced: list of column names you want to plot
# Target_combined = df['label']  # already defined

# ---- tidy data for plotting ----
plot_cols = ['label', 'DataSource'] + list(Feature_use)
D = df_filtered[plot_cols].copy()

# Optional: restore human-readable class labels
label_map = {0: 'Healthy', 1: 'Leakage'}
D['label'] = D['label'].map(label_map).astype('category')

# Optional: name your data sources
source_map = {0: 'Reference', 1: 'Monorail'}   # edit names if you like
D['DataSource'] = D['DataSource'].map(source_map).astype('category')

# Clean unusual numeric values
D.replace([np.inf, -np.inf], np.nan, inplace=True)

# Melt to long form: one row per (sample, feature)
D_long = D.melt(
    id_vars=['label','DataSource'],
    value_vars=Feature_use,
    var_name='Feature', value_name='Value'
)
D_long = D_long.dropna(subset=['Value'])

g = sns.catplot(
    data=D_long, x='label', y='Value', hue='DataSource',
    col='Feature', col_wrap=3, kind='box', height=4, sharey=False
)
g.set_axis_labels("Class label", "Value")
g.set_titles("{col_name}")

Plot the RealTime Data grouped by WV Mean Pressure

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1) Filter to DataSource == 1 ---
Data_Monorail = df_base.loc[df_base['DataSource'] == 1, ['label','WV_MeanPressure'] + list(Feature_use)].copy()

# Optional: readable class labels for x-axis
label_map = {0: 'Healthy', 1: 'Leakage'}
Data_Monorail['label'] = Data_Monorail['label'].map(label_map).astype('category')

# --- 2) Bin WV_MeanPressure into [<2, 2–3, >3] ---
edges  = [-np.inf, 2.0, 3.0, np.inf]
labels = ['< 2', '2–3', '> 3']
Data_Monorail['WV_bin'] = pd.cut(Data_Monorail['WV_MeanPressure'], bins=edges, labels=labels, right=True, include_lowest=True)

# Clean up impossible values
Data_Monorail.replace([np.inf, -np.inf], np.nan, inplace=True)

# --- 3) Melt to long form: one row per (sample, feature) ---
Data_long = Data_Monorail.melt(
    id_vars=['label','WV_bin'],
    value_vars=Feature_use,
    var_name='Feature', value_name='Value'
).dropna(subset=['Value','WV_bin'])

# --- 4) Draw grouped boxplots (hue by WV_bin) ---
sns.set(style="whitegrid")
g = sns.catplot(
    data=Data_long, x='label', y='Value', hue='WV_bin',
    col='Feature', col_wrap=3, kind='box', height=4, sharey=False
)
g.set_axis_labels("Class label", "Value")
g.set_titles("{col_name}")
g._legend.set_title("WV Mean Pressure (bar)")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1) Filter to DataSource == 1 ---
Data_Monorail = df_filtered.loc[df_base['DataSource'] == 1, ['label', 'Source'] + list(Feature_use)].copy()

# Optional: readable class labels for x-axis
label_map = {0: 'Healthy', 1: 'Leakage'}
Data_Monorail['label'] = Data_Monorail['label'].map(label_map).astype('category')

# Clean up impossible values
Data_Monorail.replace([np.inf, -np.inf], np.nan, inplace=True)

# --- 2) Melt to long form: one row per (sample, feature) ---
Data_long = Data_Monorail.melt(
    id_vars=['label', 'Source'],
    value_vars=Feature_use,
    var_name='Feature', value_name='Value'
).dropna(subset=['Value', 'Source'])

# --- 3) Draw grouped boxplots (hue by Source) ---
sns.set(style="whitegrid")
g = sns.catplot(
    data=Data_long, x='label', y='Value', hue='Source',
    col='Feature', col_wrap=3, kind='box', height=4, sharey=False
)
g.set_axis_labels("Class label", "Value")
g.set_titles("{col_name}")
g._legend.set_title("Source")
plt.show()

Plot Healthy only subset of Kit 01 and Reference

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ==============================================================
# 1) Prepare "Healthy only" subset
# ==============================================================
cols_needed = ['label', 'DataSource', 'WV_MeanPressure'] + list(Feature_use)
D = df_base[cols_needed].copy()

label_map   = {0: 'Healthy', 1: 'Leakage'}
source_map  = {0: 'Reference', 1: 'Kit-01'}
D['label']       = D['label'].map(label_map).astype('category')
D['DataSource']  = D['DataSource'].map(source_map).astype('category')

# Keep only Healthy samples
D = D[D['label'] == 'Healthy']

# ==============================================================
# 2) Bin WV_MeanPressure into [<2, 2–3, >3]
# ==============================================================
edges  = [-np.inf, 2.0, 3.0, np.inf]
labels = ['< 2', '2–3', '> 3']
D['WV_bin'] = pd.cut(
    D['WV_MeanPressure'], bins=edges, labels=labels,
    right=True, include_lowest=True
)

# ==============================================================
# 3) Loop over each feature and make a separate figure
# ==============================================================
sns.set(style="whitegrid")

for feat in Feature_use:
    plt.figure(figsize=(6, 5), dpi=300)  # increase size + resolution
    ax = sns.boxplot(
        data=D,
        x='DataSource', y=feat,
        hue='WV_bin',
        order=['Reference', 'Kit-01'],
        hue_order=['< 2', '2–3', '> 3']
    )
    ax.set_title(f"{feat} — Healthy only", fontsize=13)
    ax.set_xlabel("Data Source", fontsize=11)
    ax.set_ylabel("Value", fontsize=11)
    ax.legend(title="WV Mean Pressure (bar)")
    plt.tight_layout()
    plt.show()
    # Optionally save each plot:
    # plt.savefig(f"{feat}_Healthy_boxplot.png", dpi=300, bbox_inches='tight')


### COMBINED LEAKAGE - Variation Error of Features

In [ ]:
import numpy as np
import pandas as pd

# ---------------- safety: make sure Features_reduced is a list of strings ----------------
if isinstance(Feature_use, (str, bytes)):
    Feature_use = [Feature_use]
else:
    Feature_use = list(Feature_use)

# Keep only columns we need (and that actually exist)
base_cols = ['label', 'DataSource', 'WV_MeanPressure']
use_cols  = [c for c in Feature_use if c in df_filtered.columns]
missing   = sorted(set(Feature_use) - set(use_cols))
if missing:
    print("Warning: missing features (skipped):", missing)

D = df_filtered[base_cols + use_cols].copy()

# Map labels and sources (works whether df has 0/1 or already strings)
label_map  = {0: 'Healthy', 1: 'Leakage'}
source_map = {0: 'Reference', 1: 'Kit-01'}
D['label'] = D['label'].map(label_map).fillna(D['label'])
D['DataSource'] = D['DataSource'].map(source_map).fillna(D['DataSource'])

# Healthy only, WV in [2, 3]
D = D[(D['label'] == 'Healthy') &
      (D['WV_MeanPressure'] >= 2.0) &
      (D['WV_MeanPressure'] <= 3.0)].copy()

# Coerce features to numeric (avoid weird objects like "1.5 sec")
for c in use_cols:
    D[c] = pd.to_numeric(D[c], errors='coerce')
D.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows where ALL features are NaN (keeps rows if at least one feature is present)
D = D.dropna(subset=use_cols, how='all')

# Median per DataSource, features as rows
median_table = (
    D.groupby('DataSource')[use_cols]
      .median()
      .T
    # Ensure both columns exist; if a source is missing, you'll get NaN
    .reindex(columns=['Reference', 'Kit-01'])
)

# Compute variation: |M1 - M0| / |M0| * 100
ref = median_table['Reference']
real = median_table['Kit-01']

# avoid division by zero warnings
den = ref.replace(0, np.nan)
median_table['Variation_%'] = (real.sub(ref).abs().div(den.abs()).mul(100))

# Optional: sort by largest variation
median_table = median_table.sort_values('Variation_%', ascending=True)

print("=== Median comparison (Healthy; WV 2–3 bar) ===")
print(median_table.round(3))

median_table.round(3).to_csv('Median_Comparison_Healthy_WV2-3bar.csv', index=True)


### MANUAL BRAKE ACTIVATION - Boxchart

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

T_firstphase = df_data.copy()
# 10 Hz features to plot
vars_10hz = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

# WV_MeanPressure bins: [<2, 2–3, >3]
bins = [-np.inf, 2, 3, np.inf]
bin_labels = ['<2', '2–3', '>3']

# --- Loop over each BC_ID present in T_firstphase ---
for bc_id in T_firstphase['BC_ID'].unique():
    df_sensor = T_firstphase[T_firstphase['BC_ID'] == bc_id].copy()
    if df_sensor.empty:
        continue

    # All rows are already Standard braking
    df_sensor['brake_type'] = 'Service'

    # WV_MeanPressure bins
    wv = pd.to_numeric(df_sensor['WV_MeanPressure'], errors='coerce')
    df_sensor['wv_bin'] = pd.cut(
        wv,
        bins=bins,
        labels=bin_labels,
        right=True,        # (a,b]
        include_lowest=True
    )

    sensor_id = str(bc_id)

    # --- Figure for this sensor: 1x3 subplots ---
    fig, axes = plt.subplots(1, len(vars_10hz), figsize=(5*len(vars_10hz), 4), sharey=False)
    axes = np.atleast_1d(axes)

    for ax, var in zip(axes, vars_10hz):
        y10 = pd.to_numeric(df_sensor[var], errors='coerce')

        df_feat = pd.DataFrame({
            'value': y10,
            'brake_type': df_sensor['brake_type'],
            'wv_bin': df_sensor['wv_bin']
        }).dropna(subset=['value', 'wv_bin'])

        if df_feat.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axis('off')
            continue

        sns.boxplot(
            data=df_feat,
            x='brake_type',      # only "Standard"
            y='value',
            hue='wv_bin',        # WV_MeanPressure bins
            ax=ax
        )

        feat_name = var.replace('_', ' ')
        ax.set_title(f'{feat_name} — Sensor {sensor_id}')
        ax.set_xlabel('Braking type')
        ax.set_ylabel('Value')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    # Single legend for the whole figure
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title='WV MeanPressure bin', loc='upper right')

    # Remove legends from individual subplots
    for ax in axes:
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    fig.suptitle(f'First-phase (10 Hz) vs WV bins — Sensor {sensor_id}', y=1.05)
    fig.tight_layout()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

T_firstphase = df_data.copy()

# 10 Hz features to plot
vars_10hz = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

# WV_MeanPressure bins: [<2, 2–3, >3]
bins = [-np.inf, 2, 3, np.inf]
bin_labels = ['<2', '2–3', '>3']

# --- Loop over each BC_ID present in T_firstphase ---
for bc_id in T_firstphase['BC_ID'].unique():
    df_sensor = T_firstphase[T_firstphase['BC_ID'] == bc_id].copy()
    if df_sensor.empty:
        continue

    # WV_MeanPressure bins
    wv = pd.to_numeric(df_sensor['WV_MeanPressure'], errors='coerce')
    df_sensor['wv_bin'] = pd.cut(
        wv,
        bins=bins,
        labels=bin_labels,
        right=True,
        include_lowest=True
    )

    sensor_id = str(bc_id)
    # Map EmergencyBrake_action to descriptive labels
    df_sensor['brake_label'] = df_sensor['EmergencyBrake_action'].map({0: 'Service', 1: 'Emergency'})
    # --- Figure for this sensor: 1x3 subplots ---
    fig, axes = plt.subplots(1, len(vars_10hz), figsize=(5*len(vars_10hz), 4), sharey=False)
    axes = np.atleast_1d(axes)

    for ax, var in zip(axes, vars_10hz):
        y10 = pd.to_numeric(df_sensor[var], errors='coerce')

        df_feat = pd.DataFrame({
            'value': y10,
            'brake_label': df_sensor['brake_label'],
            'wv_bin': df_sensor['wv_bin']
        }).dropna(subset=['value', 'wv_bin', 'brake_label'])


        if df_feat.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axis('off')
            continue

        sns.boxplot(
            data=df_feat,
            x='brake_label',
            y='value',
            hue='wv_bin',
            ax=ax
        )


        feat_name = var.replace('_', ' ')
        ax.set_title(f'{feat_name} — Sensor {sensor_id}')
        ax.set_xlabel('Brake Type')
        ax.set_ylabel('Value')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    # Single legend for the whole figure
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title='WV MeanPressure bin', loc='upper right')

    # Remove legends from individual subplots
    for ax in axes:
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    fig.suptitle(f'First-phase (10 Hz) vs WV bins — Sensor {sensor_id}', y=1.05)
    fig.tight_layout()

In [ ]:
T_firstphase = df_data.copy()
vars_10hz = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

for bc_id in T_firstphase['BC_ID'].unique():
    df_sensor = T_firstphase[T_firstphase['BC_ID'] == bc_id].copy()
    if df_sensor.empty:
        continue

    # Map EmergencyBrake_action to readable labels
    df_sensor['brake_label'] = 'service'
    sensor_id = str(bc_id)

    fig, axes = plt.subplots(1, len(vars_10hz), figsize=(5*len(vars_10hz), 4), sharey=False)
    axes = np.atleast_1d(axes)

    for ax, var in zip(axes, vars_10hz):
        y10 = pd.to_numeric(df_sensor[var], errors='coerce')

        df_feat = pd.DataFrame({
            'value': y10,
            'brake_label': df_sensor['brake_label']
        }).dropna(subset=['value', 'brake_label'])

        if df_feat.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axis('off')
            continue

        sns.boxplot(
            data=df_feat,
            x='brake_label',
            y='value',
            ax=ax
        )

        feat_name = var.replace('_', ' ')
        ax.set_title(f'{feat_name} — Sensor {sensor_id}')
        ax.set_xlabel('Brake Type')
        ax.set_ylabel('Value')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    fig.suptitle(f'First-phase (10 Hz) — Sensor {sensor_id}', y=1.05)
    fig.tight_layout()

In [ ]:
# Select only sensor 0x94 from your already-filtered T_firstphase
df94 = T_firstphase[T_firstphase['BC_ID'] == '0x94'].copy()

# Features to summarize
vars_10hz = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

# Convert to numeric and compute median
median_dict = {}
for var in vars_10hz:
    median_dict[var] = pd.to_numeric(df94[var], errors='coerce').median()

# Create a table (DataFrame)
median_table = pd.DataFrame.from_dict(median_dict, orient='index', columns=['Median'])
median_table.index.name = 'Feature'

print(median_table)


# Algorithm Combined

In [ ]:
# === Unified Feature Selection Pipeline: RF, MI, ANOVA
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE, RFECV
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

# Separate features and labels
Features = df_combined.drop(['label','DataSource','Source'], axis=1)
Target = df_combined['label']
Xsel = Features.copy()

# Impute missing (with median)
imp = SimpleImputer(strategy="median")
Features = pd.DataFrame(imp.fit_transform(Features), columns=Features.columns, index=Features.index)
#Features = Features.dropna(axis=1)
# Drop columns that are all NaN or constant
Features = Features.loc[:, Features.notna().any()]  # drop all-NaN
const_mask = Features.nunique(dropna=True) <= 1
if const_mask.any():
    Features = Features.loc[:, ~const_mask]

Xsel = Features.copy()
# Keep only numeric columns (if any non-numeric slipped in)
num_cols = [c for c in Xsel.columns if np.issubdtype(Xsel[c].dtype, np.number)]
Xsel = Xsel[num_cols].copy()

# Some selectors need scaling
sc_std = StandardScaler()
sc_rob = RobustScaler()
X_std = pd.DataFrame(sc_std.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)
X_rob = pd.DataFrame(sc_rob.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)

# if y is string, change to 0/1
y_enc = pd.Series(Target).astype("category")
if y_enc.dtype.name == "category":
    y_enc = y_enc.cat.codes  # e.g., Leakage=1, Normal=0

# For stability on tiny datasets
cv = StratifiedKFold(n_splits=min(5, max(2, np.bincount(y_enc).min())), shuffle=True, random_state=42)

# Helper to convert scores to ranks (lower rank = better)
def to_rank(series, higher_is_better=True):
    s = series.copy()
    if not higher_is_better:
        s = -s
    # rank 1 = best
    return s.rank(ascending=False, method="average")

# ---------- 1) RandomForest importance ----------
# Measures a feature's utility in improving the model's prediction accuracy (e.g., mean decrease in impurity).
# Captures feature interactions naturally; highly effective.
rf = RandomForestClassifier(n_estimators=500, random_state=42, class_weight="balanced")
rf.fit(Xsel, y_enc)
rf_imp = pd.Series(rf.feature_importances_, index=Xsel.columns, name="RF_Importance")
rf_rank = to_rank(rf_imp, higher_is_better=True).rename("RF_Rank")

# ---------- 2) Mutual Information ----------
# Measures statistical dependency (information gain) between a feature and the target.
# Captures non-linear relationships. Evaluates each feature independently; ignores feature interactions.
mi = mutual_info_classif(Xsel, y_enc, random_state=42, discrete_features=False)
mi_score = pd.Series(mi, index=Xsel.columns, name="MI_Score")
mi_rank = to_rank(mi_score, higher_is_better=True).rename("MI_Rank")

# ---------- 3) ANOVA F-test ----------
# (works best if roughly Gaussian/scaled; we used imputed data)
# Measures linear correlation between a feature and the target by comparing variance between groups to variance within groups.
# Assumes linear relationship and Gaussian distribution; ignores feature interactions.
F_vals, p_vals = f_classif(Xsel, y_enc)
f_score = pd.Series(F_vals, index=Xsel.columns, name="ANOVA_F")
f_rank = to_rank(f_score, higher_is_better=True).rename("ANOVA_Rank")

# ---------- Combine all rankings ----------
rank_table = pd.concat([rf_rank, mi_rank, f_rank,
                        rf_imp, mi_score, f_score], axis=1)

# OverallRank: average of available ranks (lower = better)
rank_cols = ["RF_Rank","MI_Rank","ANOVA_Rank"]
rank_table["OverallRank"] = rank_table[rank_cols].mean(axis=1)

# Sort and display top-N
N = 5
rank_table_sorted = rank_table.sort_values("OverallRank").head(N)
print("=== Top features by OverallRank (lower = better) ===")
display(rank_table_sorted)

plt.figure(figsize=(8, max(4, 0.35*N)))
rank_table_sorted.sort_values("OverallRank")["OverallRank"].plot(kind="barh")
plt.gca().invert_yaxis()
plt.title(f"Top {N} Features by Rank")
plt.xlabel("Rank (lower is better)")
plt.tight_layout()
plt.show()

topN_features = rank_table_sorted.index.tolist()
print("\nTopN feature list:", topN_features)

Features_reduced = Features[topN_features]

# Start of Main Algorithm

## Data preparation

In [ ]:
# ======================================================================
# STEP 1: LOAD AND PREPARE DATA (REFERENCE + MULTI-KIT MONORAIL)
# ======================================================================

def load_data(model_path, monorail_paths):
    """
    Load and prepare reference (model) data and Monorail data (one or more kits),
    align common columns, and return a single combined DataFrame.

    Parameters
    ----------
    model_path : str
        Path to model.csv (reference experimental campaign).
    monorail_paths : str or list of str
        Path or list of paths to Monorail TestBrakefinal_data_kitXX.csv files.

    Returns
    -------
    df_base : pandas.DataFrame
        Combined DataFrame with:
        - aligned common columns between reference and Monorail,
        - binary label (0/1) where available,
        - 'Source' column (kit ID or 0 for reference),
        - 'DataSource' column (0 = reference, 1 = Monorail).
    """

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking
        if 'Non_Standard_Braking' in df.columns:
            df = df[df['Non_Standard_Braking'] == 0]

        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_kit06.csv" -> 6
        match = re.search(r'kit(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                # AttributeError if column is not string-like; ValueError if some values cannot be cast
                continue

        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # Binary label from malfunction code
    leakage_codes = ['C', 'D', 'E', 'F', 'G']
    df_reference['LeakageLabel'] = np.where(
        df_reference['Malfunction'].isin(leakage_codes),
        'Combined leakage',
        'Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        # If any of these columns are missing in some version of model.csv, guard with .get
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    # Drop original per-phase columns (only those that actually exist)
    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    # Ensure 'Source' is integer in both
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)

    # Columns common to BOTH datasets
    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    # Subsets with only common columns + a DataSource flag
    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0  # 0 = reference campaign

    df_data_subset = df_data[common_cols].copy()
    df_data_subset['DataSource'] = 1       # 1 = Monorail (real-time) data

    # Stack reference + Monorail
    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # Encode final label column (will be NaN for Monorail if it has no LeakageLabel)
    if 'LeakageLabel' in df_combined.columns:
        df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue
    df_combined["WV_bin"] = df_combined["WV_MeanPressure"].apply(
    lambda p: np.nan if pd.isna(p) else (0 if p < 2 else (2 if p > 3 else 1))
    )
    df_base = df_combined.copy()
    return df_base

model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_kit01.csv',
    'TestBrakefinal_data_kit06.csv',
    'TestBrakefinal_data_kit27.csv'
]

df = load_data(model_path, monorail_paths)

print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail


In [ ]:
# ======================================================================
# STEP 2: PREPROCESS DATA FOR ML
# ======================================================================

from sklearn.model_selection import train_test_split

def preprocess_data(df, features, test_size=0.2):
    """
    Preprocess data: create WV_bin column, filter by bin==1, select features,
    split into train/test, and separate healthy samples for training.
    
    Parameters:
    - df: pandas DataFrame containing the data
    - features: list of feature names to use (e.g., ['Total_power_efficiency', 'FlowRate'])
    - test_size: proportion of the dataset to include in the test split
    
    Returns:
    - X_train, X_test, y_train, y_test, X_train_healthy
    """
    
    # Create WV_bin column
    df = df.copy()
    # Filter by WV_bin == 1 (pressure between 2 and 3)
    df_filt = df[df["WV_bin"] == 1].copy()
    
    # Validate feature selection
    missing = [f for f in features if f not in df_filt.columns]
    if missing:
        raise ValueError(f"The following features are not in the dataframe: {missing}")
    
    # Select features and labels
    X = df_filt[features]
    y = df_filt['label']
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )
    
    # Extract healthy training samples
    X_train_healthy = X_train[y_train == 0]
    
    print(f"Total samples: {len(df)}")
    print(f"Filtered samples (WV_bin==1): {len(df_filt)}")
    print(f"Training samples: {len(X_train)} (Healthy: {sum(y_train==0)}, Leakage: {sum(y_train==1)})")
    print(f"Training samples (healthy only): {len(X_train_healthy)}")
    print(f"Test samples: {len(X_test)} (Healthy: {sum(y_test==0)}, Leakage: {sum(y_test==1)})")
    
    return X_train, X_test, y_train, y_test, X_train_healthy

## Model Definition

### Defining function for Model used, Train Model, and Cross Validations

In [ ]:
# ============================================================================
# STEP 1: DEFINE ALL MODELS
# ============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from xgboost import XGBClassifier

def get_all_models(contamination=0.05):
    """
    Define all models to be tested
    Returns dictionary of models grouped by type
    """
    models = {

        # =====================================================================
        # 1. ANOMALY DETECTION MODELS
        # =====================================================================
        'anomaly': {
            'Isolation Forest': IsolationForest(
                contamination=contamination,
                random_state=42,
                n_estimators=100,
                n_jobs=-1
            ),
            'One-Class SVM': OneClassSVM(
                nu=contamination,
                kernel='rbf',
                gamma='auto'
            ),
            'Local Outlier Factor': LocalOutlierFactor(
                contamination=contamination,
                novelty=True,
                n_neighbors=25
            )
        },

        # =====================================================================
        # 2. SUPERVISED MODELS (IMBALANCED DATA)
        # =====================================================================
        'supervised_imbalanced': {
            # Your existing models
            'Random Forest (Weighted)': RandomForestClassifier(
                class_weight='balanced',
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            ),
            'XGBoost (Weighted)': XGBClassifier(
                scale_pos_weight=(1100/23),  
                n_estimators=100,
                random_state=42,
                eval_metric='logloss'
            ),
            'Logistic Regression (Weighted)': LogisticRegression(
                class_weight='balanced',
                random_state=42,
                max_iter=1000,
                solver='liblinear'
            ),
            'Decision Tree (Weighted)': DecisionTreeClassifier(
                class_weight='balanced',
                random_state=42,
                max_depth=1
            ),

            # NEW: KNN classifier
            'KNN (Imbalanced)': KNeighborsClassifier(
                n_neighbors=5,       # will be tuned
                weights='distance',
                metric='minkowski',
                p=2
            ),
        },

        # =====================================================================
        # 3. SUPERVISED MODELS (WITH SMOTE)
        # =====================================================================
        'supervised_smote': {
            'Random Forest (SMOTE)': RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            ),
            'XGBoost (SMOTE)': XGBClassifier(
                n_estimators=200,
                random_state=42,
                eval_metric='logloss'
            ),

            # NEW: Logistic Regression (SMOTE version)
            'Logistic Regression (SMOTE)': LogisticRegression(
                random_state=42,
                max_iter=1000,
                solver='liblinear'
            ),

            # Already present Decision Tree
            'Decision Tree (SMOTE)': DecisionTreeClassifier(
                random_state=42,
                max_depth=1
            ),

            # NEW: KNN with SMOTE
            'KNN (SMOTE)': KNeighborsClassifier(
                n_neighbors=5,
                weights='distance',
                metric='minkowski',
                p=2
            )
        }
    }

    return models


# ============================================================================
# STEP 2: TRAIN ALL MODELS
# ============================================================================

def train_all_models(X_train_scaled, y_train, X_train_healthy_scaled, contamination=0.05):
    """
    Train all models and return trained models with metadata
    """
    models = get_all_models(contamination)
    trained_models = {}
    
    print("\n" + "="*60)
    print("TRAINING ALL MODELS")
    print("="*60)
    
    # Train anomaly detection models (on healthy data only)
    print("\n[1/3] Training Anomaly Detection Models...")
    for name, model in models['anomaly'].items():
        print(f"  - Training {name}...")
        model.fit(X_train_healthy_scaled)
        trained_models[name] = {
            'model': model,
            'type': 'anomaly',
            'trained': True
        }
    
    # Train supervised models on imbalanced data
    print("\n[2/3] Training Supervised Models (Imbalanced Data)...")
    for name, model in models['supervised_imbalanced'].items():
        print(f"  - Training {name}...")
        model.fit(X_train_scaled, y_train)
        trained_models[name] = {
            'model': model,
            'type': 'supervised',
            'trained': True
        }
    
    # Apply SMOTE and train supervised models
    print("\n[3/3] Training Supervised Models (with SMOTE)...")
    smote = SMOTE(random_state=42)
    X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
    print(f"  - After SMOTE: {sum(y_train_smote==0)} healthy, {sum(y_train_smote==1)} leakage")
    
    for name, model in models['supervised_smote'].items():
        print(f"  - Training {name}...")
        model.fit(X_train_smote, y_train_smote)
        trained_models[name] = {
            'model': model,
            'type': 'supervised_smote',
            'trained': True
        }
    
    print("\nAll models trained successfully!")
    return trained_models


In [ ]:
# ============================================================================
# STEP 3: CROSS VALIDATION METHOD
# ============================================================================

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone
import pandas as pd
import numpy as np

def crossvalidated_metrics_table(trained_models, X, y):
    X = np.asarray(X)
    y = np.asarray(y)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    rows = []

    for name, entry in trained_models.items():
        base_model = entry['model']
        mtype      = entry['type']

        # ---- Build estimator properly ----
        if mtype == "supervised":
            estimator = clone(base_model)

        elif mtype == "supervised_smote":
            estimator = ImbPipeline([
                ('smote', SMOTE(random_state=42)),
                ('clf', clone(base_model))
            ])

        elif mtype == "anomaly":
            # anomaly models need tuned threshold
            threshold = entry.get("threshold", None)
            y_pred = np.zeros_like(y)

            for train_idx, test_idx in cv.split(X, y):
                X_train, X_test = X[train_idx], X[test_idx]
                y_train = y[train_idx]

                X_train_healthy = X_train[y_train == 0]

                model = clone(base_model)
                model.fit(X_train_healthy)

                scores = model.decision_function(X_test)

                # apply tuned threshold if available
                if threshold is not None:
                    y_pred[test_idx] = (scores < threshold).astype(int)
                else:
                    # fallback to IF default threshold
                    out = model.predict(X_test)
                    y_pred[test_idx] = (out == -1).astype(int)

        else:
            continue

        # ---- Supervised models → get out-of-fold predictions ----
        if mtype != "anomaly":
            y_pred_proba = cross_val_predict(
                estimator, X, y, cv=cv, method='predict_proba'
            )[:, 1]
            y_pred = (y_pred_proba >= 0.5).astype(int)

        # ---- confusion matrix ----
        tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

        # ---- metrics ----
        precision = precision_score(y, y_pred, zero_division=0)
        recall    = recall_score(y, y_pred, zero_division=0)
        f1        = f1_score(y, y_pred, zero_division=0)

        # ROC-AUC for supervised
        if mtype != "anomaly":
            rocauc = roc_auc_score(y, y_pred_proba)
        else:
            rocauc = np.nan

        rows.append({
            "Model": name,
            "Type": mtype,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": rocauc,
            "True Positives": tp,
            "False Positives": fp,
            "False Negatives": fn,
            "True Negatives": tn,
        })

    df = pd.DataFrame(rows).sort_values(by="F1-Score", ascending=False)
    return df

### Model Hyperparameter Tuning

For random forest and XGBoost

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

def tune_random_forest_weighted(X_train_scaled, y_train):
    """
    Hyperparameter tuning for Random Forest (Weighted).
    Returns best estimator, best_params, best_score.
    """
    rf_base = RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    param_grid = {
        'n_estimators':      [100, 200, 500, 800],
        'max_depth':         [3, 5, 7, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
        'max_features':      ['sqrt', 'log2', 0.5],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=rf_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    print("\n===== Tuning Random Forest (Weighted) =====")
    grid.fit(X_train_scaled, y_train)

    print("Best RF params:", grid.best_params_)
    print("Best RF CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_


def tune_xgboost_weighted(X_train_scaled, y_train):
    """
    Hyperparameter tuning for XGBoost (Weighted).
    Assumes binary labels {0: healthy, 1: leakage}.
    Returns best estimator, best_params, best_score.
    """
    # Base model (scale_pos_weight will be tuned)
    xgb_base = XGBClassifier(
        eval_metric='logloss',
        random_state=42,
        n_estimators=200,
        use_label_encoder=False # optional depending on xgboost version
    )

    # compute approximate neg/pos ratio:
    n_pos = (y_train == 1).sum()
    n_neg = (y_train == 0).sum()
    ratio = n_neg / max(n_pos, 1)

    param_grid = {
        'n_estimators':      [100, 200, 400],
        'learning_rate':     [0.01, 0.05, 0.1],
        'max_depth':         [2, 3, 4, 5],
        'min_child_weight':  [1, 3, 5],
        'subsample':         [0.6, 0.8, 1.0],
        'colsample_bytree':  [0.6, 0.8, 1.0],
        'gamma':             [0.0, 0.1, 0.5],
        # tune around the empirical imbalance ratio
        'scale_pos_weight':  [0.5*ratio, ratio, 2*ratio],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=xgb_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    print("\n===== Tuning XGBoost (Weighted) =====")
    grid.fit(X_train_scaled, y_train)

    print("Best XGB params:", grid.best_params_)
    print("Best XGB CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_

Logistic Regression Tuning

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_logistic_regression_weighted(X_train_scaled, y_train):
    """
    Hyperparameter tuning for Logistic Regression (Weighted).
    Uses class_weight='balanced' to handle class imbalance.
    Returns best estimator, best_params, best_score.
    """

    lr_base = LogisticRegression(
        class_weight='balanced',
        solver='liblinear',   # robust for small datasets
        max_iter=500,
        random_state=42
    )

    param_grid = {
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=lr_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    print("\n===== Tuning Logistic Regression (Weighted) =====")
    grid.fit(X_train_scaled, y_train)

    print("Best LR params:", grid.best_params_)
    print("Best LR CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_


KNN Tuning

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

def tune_knn_classifier(X_train_scaled, y_train):
    """
    Hyperparameter tuning for K-Nearest Neighbors classifier.
    Returns best estimator, best_params, best_score.
    """

    knn_base = KNeighborsClassifier()

    param_grid = {
        'n_neighbors': [3, 5, 7, 9, 11, 15],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan', 'minkowski'],
        'p': [1, 2]   # p=1 (Manhattan), p=2 (Euclidean)
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=knn_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    print("\n===== Tuning KNN Classifier =====")
    grid.fit(X_train_scaled, y_train)

    print("Best KNN params:", grid.best_params_)
    print("Best KNN CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_


Decision Tree tuning

In [ ]:
from sklearn.tree import DecisionTreeClassifier

def tune_decision_tree_weighted(X_train_scaled, y_train):
    """
    Hyperparameter tuning for Decision Tree (Weighted).
    Returns best estimator, best_params, best_score.
    """

    dt_base = DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42
    )

    param_grid = {
        'criterion': ['gini', 'entropy', 'log_loss'],
        'max_depth': [1, 3, 5, 7, 10, 15],
        'min_samples_split': [2, 5, 10, 20],
        'min_samples_leaf': [1, 2, 4, 6],
        'max_features': ['sqrt', 'log2', None]
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=dt_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    print("\n===== Tuning Decision Tree (Weighted) =====")
    grid.fit(X_train_scaled, y_train)

    print("Best DT params:", grid.best_params_)
    print("Best DT CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_


## Algorithm 1 - Use only TPE

In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING AND FEATURE SELECTION
# ============================================================================
selected_features = ['Total_power_efficiency']

[X_train_tpe, X_test_tpe, y_train, y_test, X_train_healthy_tpe] = preprocess_data(df, selected_features)
X_train_tpe.head()

In [ ]:
# ============================================================================
# STEP 2: IMPUTE + FEATURE SCALING
# ============================================================================

from sklearn.impute import SimpleImputer

def scale_features(X_train, X_test, X_train_healthy):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    X_train_healthy_imp = imputer.transform(X_train_healthy)

    # 2) Standardization (fit only on imputed training set)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_train_healthy_scaled = scaler.transform(X_train_healthy_imp)

    return X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer

[X_train_scaled_tpe, X_test_scaled_tpe, X_train_healthy_scaled_tpe, scaler, imputer] = scale_features(X_train_tpe, X_test_tpe, X_train_healthy_tpe)

### Train the Model

In [ ]:
trained_models = train_all_models(X_train_scaled_tpe, y_train, X_train_healthy_scaled_tpe, contamination=0.05)

### Plot Results

Cross Validate only on training set

In [ ]:
cv_summary_tpe = crossvalidated_metrics_table(
    trained_models, X_train_tpe, y_train
)

cv_summary_tpe

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Define the models to visualize (all are supervised)
best_models = [
    "KNN (Imbalanced)",
    "Logistic Regression (SMOTE)",
    "Logistic Regression (Weighted)",
    "Decision Tree (Weighted)",
    "Random Forest (Weighted)",
    "XGBoost (Weighted)"
]

# Use the TEST set only for a single final evaluation (no CV here)
X = np.asarray(X_test_tpe)
y = np.asarray(y_test)

# Create subplots: 1 row per model, 2 columns (raw + normalized)
fig_height = len(best_models) * 2.5  # 2.5 inches per model row
fig, axes = plt.subplots(len(best_models), 2, figsize=(12, fig_height))

for idx, model_name in enumerate(best_models):
    model_info = trained_models[model_name]
    mtype = model_info['type']

    if mtype not in ('supervised', 'supervised_smote'):
        raise ValueError(f"{model_name} is not a supervised model.")

    # Use the model trained earlier on the TRAINING data
    base_model = model_info['model']

    # NO cross_val_predict here — just direct prediction on the test set
    y_pred = base_model.predict(X)

    # Confusion matrix
    cm = confusion_matrix(y, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    # Plot raw counts
    ax_raw = axes[idx, 0]
    im_raw = ax_raw.imshow(cm, cmap='coolwarm')
    ax_raw.set_title(f'{model_name} (Counts)')
    ax_raw.set_xticks([0, 1]); ax_raw.set_yticks([0, 1])
    ax_raw.set_xticklabels(['Pred Healthy', 'Pred Leakage'])
    ax_raw.set_yticklabels(['True Healthy', 'True Leakage'])
    for i in range(2):
        for j in range(2):
            text_color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax_raw.text(j, i, cm[i, j], ha='center', va='center', color=text_color)
    plt.colorbar(im_raw, ax=ax_raw, fraction=0.046, pad=0.04)

    # Plot normalized
    ax_norm = axes[idx, 1]
    im_norm = ax_norm.imshow(cm_norm, cmap='coolwarm', vmin=0, vmax=1)
    ax_norm.set_title(f'{model_name} (Normalized)')
    ax_norm.set_xticks([0, 1]); ax_norm.set_yticks([0, 1])
    ax_norm.set_xticklabels(['Pred Healthy', 'Pred Leakage'])
    ax_norm.set_yticklabels(['True Healthy', 'True Leakage'])
    for i in range(2):
        for j in range(2):
            text_color = 'white' if cm_norm[i, j] > 0.5 else 'black'
            ax_norm.text(j, i, f"{cm_norm[i, j]:.2f}", ha='center', va='center', color=text_color)
    plt.colorbar(im_norm, ax=ax_norm, fraction=0.046, pad=0.04)

# Final layout
plt.suptitle('Confusion Matrices for Top Models (Test Set)', y=1.02)
plt.tight_layout()
for ax in axes.flatten():
    ax.grid(False)
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import cross_val_predict

plt.figure(figsize=(7, 6))

models_to_plot = [
    "Random Forest (Weighted)",
    "Decision Tree (SMOTE)",
    "Logistic Regression (Weighted)",
    "XGBoost (SMOTE)",
    "KNN (Imbalanced)"
]

X = np.asarray(X_train_scaled_tpe)
y = np.asarray(y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name in models_to_plot:
    entry = trained_models[name]
    base_model = entry['model']
    mtype      = entry['type']

    # Build estimator
    if mtype == 'supervised':
        estimator = clone(base_model)
    elif mtype == 'supervised_smote':
        estimator = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', clone(base_model))
        ])
    else:
        print(f"Skipping anomaly model: {name}")
        continue

    # CV predicted probabilities
    y_scores = cross_val_predict(
        estimator, X, y,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    fpr, tpr, _ = roc_curve(y, y_scores)
    auc = roc_auc_score(y, y_scores)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", lw=2)

# Random baseline
plt.plot([0, 1], [0, 1], 'k--', label="Random classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Cross-validated ROC Curves for Multiple Models")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


### Metrics - F1 Score Table

In [ ]:
cv_summary_tpe

In [ ]:
from sklearn.tree import export_text

tree = trained_models['Decision Tree (Weighted)']['model']
print(export_text(tree, feature_names=['TPE']))

original_threshold = scaler.inverse_transform([[-1.12, 0]])  # 0 is a placeholder for TPD
print(original_threshold[0][0])  # Extract the TPE value


### Hyperparameter Tuning of 1 Features

In [ ]:
best_lr, lr_params, lr_f1 = tune_logistic_regression_weighted(X_train_scaled_tpe, y_train)
best_knn, knn_params, knn_f1 = tune_knn_classifier(X_train_scaled_tpe, y_train)
best_dt, dt_params, dt_f1 = tune_decision_tree_weighted(X_train_scaled_tpe, y_train)

In [ ]:
# Run XGboost tuning
best_xgb_est_tpe, best_xgb_params_tpe, best_xgb_score_tpe = tune_xgboost_weighted(
    X_train_scaled_tpe, y_train
)

In [ ]:
# Run Random Forest tuning
best_rf_est_tpe, best_rf_params_tpe, best_rf_score_tpe = tune_random_forest_weighted(
    X_train_scaled_tpe, y_train
)

In [ ]:
# Replace only the model with the best estimator
tuned_models = trained_models.copy()
tuned_models['Logistic Regression (Weighted)']['model'] = best_lr
tuned_models['KNN (Imbalanced)']['model'] = best_knn
tuned_models['Decision Tree (Weighted)']['model'] = best_dt
tuned_models['XGBoost (Weighted)']['model'] = best_xgb_est_tpe
tuned_models['Random Forest (Weighted)']['model'] = best_rf_est_tpe

# Evaluate
cv_results_1_feat_tuned = crossvalidated_metrics_table(
    tuned_models, X_train_scaled_tpe, y_train
)
cv_results_1_feat_tuned

In [ ]:
import numpy as np

rf = tuned_models['KNN (Imbalanced)']['model']
probs = rf.predict_proba(X_train_scaled_tpe)[:, 1]

# ROC curve gives candidate thresholds
from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(y_train, probs)

# Find threshold that maximizes F1 on training set
f1_scores = 2 * precision * recall / (precision + recall)
best_idx = np.nanargmax(f1_scores)
best_threshold = thresholds[best_idx]
# 1. Extract all unique TPE values from training data
tpe_vals = np.sort(X_train_scaled_tpe.ravel())

# 2. Create a very fine grid for smoother threshold detection
grid = np.linspace(tpe_vals.min(), tpe_vals.max(), 500)

# 3. Predict probabilities for all grid points
probs = rf.predict_proba(grid.reshape(-1, 1))[:, 1]

# 4. Find where probability crosses the chosen threshold
idx = np.argmin(np.abs(probs - best_threshold))
tpe_threshold_scaled = grid[idx]
# Pad with dummy value for the second feature
tpe_threshold_original = scaler.inverse_transform([[tpe_threshold_scaled, 0]])[0][0]

print("Best F1 threshold =", best_threshold)
print("Scaled TPE threshold =", tpe_threshold_scaled)
print("Original TPE threshold =", tpe_threshold_original)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# Get the trained decision tree model
tree = tuned_models['Decision Tree (Weighted)']['model']

# Plot the tree
plt.figure(figsize=(16, 10))  # Adjust size as needed
plot_tree(
    tree,
    feature_names=['TPE'],
    class_names=['Healthy', 'Leakage'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree Visualization")
plt.show()

## Feature Sensitivity Analysis based on F1 Scores

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, make_scorer
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Union
import matplotlib.pyplot as plt
import seaborn as sns


def feature_sensitivity_analysis(
    X: Union[pd.DataFrame, np.ndarray],
    y: np.ndarray,
    feature_names: List[str],
    models: Dict[str, object],
    max_features: Optional[int] = None,
    cv_splits: int = 5,
    scoring: str = "f1",
    step: int = 1,
    random_state: int = 42,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Perform incremental feature sensitivity analysis across multiple models.
    
    Tests model performance as features are progressively added, helping identify
    the optimal feature subset size and compare model behavior.
    
    Parameters
    ----------
    X : pd.DataFrame or np.ndarray
        Feature matrix with all candidate features
    y : np.ndarray
        Target labels (binary or multiclass)
    feature_names : List[str]
        Ordered feature names (typically sorted by importance)
    models : Dict[str, estimator]
        Dictionary mapping model names to sklearn estimators
    max_features : int, optional
        Maximum features to test. Defaults to all features
    cv_splits : int, default=5
        Number of cross-validation folds
    scoring : str, default="f1"
        Scoring metric (currently supports 'f1', 'accuracy', 'roc_auc')
    step : int, default=1
        Increment size for feature count (use >1 for faster analysis)
    random_state : int, default=42
        Random seed for reproducibility
    verbose : bool, default=True
        Print progress information
        
    Returns
    -------
    pd.DataFrame
        Results with columns: Model, NumFeatures, MeanScore, StdScore, Features
    """
    
    # Validate inputs
    if max_features is None:
        max_features = len(feature_names)
    max_features = min(max_features, len(feature_names))
    
    # Setup CV and scorer
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    scorer, score_label = _get_scorer(scoring)
    
    results = []
    
    for model_name, base_estimator in models.items():
        if verbose:
            print(f"\n{'='*50}")
            print(f"Analyzing: {model_name}")
            print(f"{'='*50}")
        
        for k in range(1, max_features + 1, step):
            subset_features = feature_names[:k]
            X_subset = _subset_features(X, subset_features, feature_names)
            
            # Cross-validation
            estimator = clone(base_estimator)
            scores = cross_val_score(
                estimator, X_subset, y, 
                cv=cv, 
                scoring=scorer,
                n_jobs=-1  # Parallel processing
            )
            
            results.append({
                "Model": model_name,
                "NumFeatures": k,
                "MeanScore": np.mean(scores),
                "StdScore": np.std(scores),
                "Features": ", ".join(subset_features[:3]) + (f" (+{k-3} more)" if k > 3 else "")
            })
            
            if verbose and k % 5 == 0:
                print(f"  {k:3d} features: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
    
    results_df = pd.DataFrame(results)
    
    if verbose:
        print(f"\n{'='*50}")
        print("Analysis Complete!")
        print(f"{'='*50}\n")
        _print_summary(results_df)
    
    return results_df


def _get_scorer(scoring: str):
    """Get appropriate scorer and label."""
    scorers = {
        "f1": (make_scorer(f1_score, average="binary"), "F1"),
        "accuracy": ("accuracy", "Accuracy"),
        "roc_auc": ("roc_auc", "ROC-AUC")
    }
    
    if scoring not in scorers:
        raise ValueError(f"Scoring '{scoring}' not supported. Use: {list(scorers.keys())}")
    
    return scorers[scoring]


def _subset_features(X, subset_features, all_feature_names):
    """Extract feature subset from X."""
    if isinstance(X, pd.DataFrame):
        return X[subset_features].values
    else:
        indices = [all_feature_names.index(f) for f in subset_features]
        return X[:, indices]


def _print_summary(df: pd.DataFrame):
    """Print summary of best results per model."""
    print("Best Performance by Model:")
    print("-" * 70)
    
    for model in df["Model"].unique():
        model_df = df[df["Model"] == model]
        best_idx = model_df["MeanScore"].idxmax()
        best = model_df.loc[best_idx]
        
        print(f"{model:20s} | {best['NumFeatures']:3d} features | "
              f"Score: {best['MeanScore']:.4f} ± {best['StdScore']:.4f}")


def plot_sensitivity_analysis(
    results_df: pd.DataFrame,
    figsize: tuple = (12, 6),
    save_path: Optional[str] = None
):
    """
    Create visualization of feature sensitivity analysis results.
    
    Parameters
    ----------
    results_df : pd.DataFrame
        Output from feature_sensitivity_analysis()
    figsize : tuple, default=(12, 6)
        Figure size
    save_path : str, optional
        Path to save the figure
    """
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # Plot 1: Mean score with confidence bands
    for model in results_df["Model"].unique():
        model_data = results_df[results_df["Model"] == model]
        
        ax1.plot(
            model_data["NumFeatures"], 
            model_data["MeanScore"],
            marker='o', 
            label=model, 
            linewidth=2
        )
        ax1.fill_between(
            model_data["NumFeatures"],
            model_data["MeanScore"] - model_data["StdScore"],
            model_data["MeanScore"] + model_data["StdScore"],
            alpha=0.2
        )
    
    ax1.set_xlabel("Number of Features", fontsize=12)
    ax1.set_ylabel("Mean Score", fontsize=12)
    ax1.set_title("Model Performance vs Feature Count", fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Score at optimal feature count
    best_scores = []
    for model in results_df["Model"].unique():
        model_data = results_df[results_df["Model"] == model]
        best_idx = model_data["MeanScore"].idxmax()
        best = model_data.loc[best_idx]
        best_scores.append({
            "Model": model,
            "Score": best["MeanScore"],
            "NumFeatures": best["NumFeatures"]
        })
    
    best_df = pd.DataFrame(best_scores)
    
    bars = ax2.barh(best_df["Model"], best_df["Score"])
    for i, (score, n_feat) in enumerate(zip(best_df["Score"], best_df["NumFeatures"])):
        ax2.text(score + 0.01, i, f'{score:.3f}\n({n_feat} feat)', 
                va='center', fontsize=10)
    
    ax2.set_xlabel("Best Score", fontsize=12)
    ax2.set_title("Peak Performance Comparison", fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()


# Example usage:
"""
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42)
}

# Assuming feature_names is sorted by importance
results = feature_sensitivity_analysis(
    X=X_train,
    y=y_train,
    feature_names=sorted_features,
    models=models,
    max_features=20,
    step=1  # Test every feature count
)

# Visualize
plot_sensitivity_analysis(results, save_path='sensitivity_analysis.png')
"""

In [ ]:
clean_models = {
    name: d["model"]        # extract just the estimator
    for name, d in trained_models.items()
}
# X, y already defined; features already filtered (e.g. WV between 2–3 bar) in your current pipeline
ordered_features = [
    "Total_power_efficiency",
    "Total_power_delay",
    "Buildup_end_pressure_delay",
    "Release_start_pressure_delay",
    "Total_energy_efficiency",   
]
selected_models = {
    "KNN (Imbalanced)": clean_models["KNN (Imbalanced)"],
    "Logistic Regression (Weighted)": clean_models["Logistic Regression (Weighted)"],
    "Decision Tree (Weighted)": clean_models["Decision Tree (Weighted)"],
}

[X_train, X_test, y_train, y_test, X_train_healthy] = preprocess_data(df, ordered_features)

[X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer] = scale_features(X_train, X_test, X_train_healthy)

In [ ]:
# 2) Run sensitivity analysis
results_df = feature_sensitivity_analysis(
    X=X_train_scaled,
    y=y_train,
    feature_names=ordered_features,
    models=selected_models,
    max_features=20,
    step=1  # Test every feature count
)

plot_sensitivity_analysis(results_df)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import spearmanr


def diagnose_feature_issues(X, feature_names, y=None):
    """
    Comprehensive diagnostics for feature interaction issues.
    
    Identifies:
    - Feature correlations (multicollinearity)
    - Scale differences
    - Information redundancy
    - Feature importance to target
    """
    
    if isinstance(X, np.ndarray):
        X_df = pd.DataFrame(X, columns=feature_names)
    else:
        X_df = X[feature_names].copy()
    
    print("="*70)
    print("FEATURE DIAGNOSTICS")
    print("="*70)
    
    # 1. Scale Analysis
    print("\n1. FEATURE SCALES (Raw Statistics)")
    print("-"*70)
    scale_stats = X_df.describe().T[['mean', 'std', 'min', 'max']]
    scale_stats['range'] = scale_stats['max'] - scale_stats['min']
    scale_stats['coef_var'] = scale_stats['std'] / scale_stats['mean'].abs()
    print(scale_stats)
    print("\n⚠️  Large differences in 'range' or 'std' can dominate KNN distances!")
    
    # 2. Correlation Analysis
    print("\n\n2. FEATURE CORRELATIONS (Pearson)")
    print("-"*70)
    corr_matrix = X_df.corr()
    print(corr_matrix)
    
    # Find high correlations
    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_val = abs(corr_matrix.iloc[i, j])
            if corr_val > 0.7:  # Threshold for concern
                high_corr_pairs.append({
                    'Feature 1': corr_matrix.columns[i],
                    'Feature 2': corr_matrix.columns[j],
                    'Correlation': corr_val
                })
    
    if high_corr_pairs:
        print("\n⚠️  HIGH CORRELATIONS DETECTED (>0.7):")
        for pair in high_corr_pairs:
            print(f"   {pair['Feature 1']} ↔ {pair['Feature 2']}: {pair['Correlation']:.3f}")
        print("\n   → This creates redundancy and hurts KNN performance!")
    else:
        print("\n✓ No concerning correlations found")
    
    # 3. Feature Importance to Target (if provided)
    if y is not None:
        print("\n\n3. FEATURE-TARGET RELATIONSHIP (Spearman Correlation)")
        print("-"*70)
        target_corrs = []
        for feat in feature_names:
            corr, pval = spearmanr(X_df[feat], y)
            target_corrs.append({
                'Feature': feat,
                'Correlation': abs(corr),
                'P-value': pval
            })
        
        target_df = pd.DataFrame(target_corrs).sort_values('Correlation', ascending=False)
        print(target_df.to_string(index=False))
        
        weak_features = target_df[target_df['Correlation'] < 0.3]['Feature'].tolist()
        if weak_features:
            print(f"\n⚠️  Weak features (|corr| < 0.3): {weak_features}")
    
    # 4. Variance Inflation Factor (VIF) - Multicollinearity
    print("\n\n4. MULTICOLLINEARITY CHECK (VIF)")
    print("-"*70)
    print("(VIF > 10 indicates severe multicollinearity)")
    
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    
    vif_data = []
    for i, col in enumerate(feature_names):
        vif = variance_inflation_factor(X_df.values, i)
        vif_data.append({'Feature': col, 'VIF': vif})
    
    vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)
    print(vif_df.to_string(index=False))
    
    high_vif = vif_df[vif_df['VIF'] > 10]['Feature'].tolist()
    if high_vif:
        print(f"\n⚠️  High VIF features: {high_vif}")
        print("   → Consider removing these for KNN!")
    
    # 5. Visualization
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # Correlation heatmap
    ax1 = fig.add_subplot(gs[0:2, 0:2])
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
    
    # Scale comparison
    ax2 = fig.add_subplot(gs[0, 2])
    X_scaled = StandardScaler().fit_transform(X_df)
    scale_comparison = pd.DataFrame({
        'Raw': X_df.std(),
        'Scaled': pd.DataFrame(X_scaled, columns=feature_names).std()
    })
    scale_comparison.plot(kind='bar', ax=ax2)
    ax2.set_title('Scale Before/After Standardization', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Standard Deviation')
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # VIF comparison
    ax3 = fig.add_subplot(gs[1, 2])
    colors = ['red' if v > 10 else 'orange' if v > 5 else 'green' for v in vif_df['VIF']]
    ax3.barh(vif_df['Feature'], vif_df['VIF'], color=colors)
    ax3.axvline(x=10, color='red', linestyle='--', label='Critical (>10)')
    ax3.axvline(x=5, color='orange', linestyle='--', label='Warning (>5)')
    ax3.set_xlabel('VIF Score')
    ax3.set_title('Multicollinearity (VIF)', fontsize=12, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis='x')
    
    # PCA variance explained
    ax4 = fig.add_subplot(gs[2, :])
    pca = PCA()
    pca.fit(StandardScaler().fit_transform(X_df))
    cumsum = np.cumsum(pca.explained_variance_ratio_)
    
    ax4.plot(range(1, len(cumsum)+1), cumsum, 'bo-', linewidth=2, markersize=8)
    ax4.axhline(y=0.95, color='r', linestyle='--', label='95% variance')
    ax4.set_xlabel('Number of Components', fontsize=12)
    ax4.set_ylabel('Cumulative Explained Variance', fontsize=12)
    ax4.set_title('PCA Analysis: Information Redundancy', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    ax4.legend()
    
    # Add text annotation
    n_components_95 = np.argmax(cumsum >= 0.95) + 1
    ax4.text(n_components_95, 0.95, f'  {n_components_95} components\n  capture 95% variance',
             verticalalignment='bottom', fontsize=10, color='red')
    
    plt.tight_layout()
    plt.show()
    
    # 6. Recommendations
    print("\n\n" + "="*70)
    print("RECOMMENDATIONS FOR KNN")
    print("="*70)
    
    recommendations = []
    
    if high_corr_pairs:
        recommendations.append(
            "🔴 Remove redundant features: Features with >0.7 correlation carry duplicate information"
        )
    
    if high_vif:
        recommendations.append(
            f"🔴 Address multicollinearity: Remove or combine {high_vif}"
        )
    
    if scale_stats['range'].max() / scale_stats['range'].min() > 100:
        recommendations.append(
            "🟡 ALWAYS use StandardScaler for KNN - scale differences are extreme"
        )
    
    if n_components_95 < len(feature_names):
        recommendations.append(
            f"🟡 Consider using only {n_components_95} features (95% of information)"
        )
    
    if not recommendations:
        recommendations.append("✓ Features look reasonable for KNN")
    
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec}")
    
    return {
        'correlation_matrix': corr_matrix,
        'vif_scores': vif_df,
        'scale_stats': scale_stats,
        'high_correlations': high_corr_pairs,
        'n_components_95': n_components_95
    }


def compare_feature_subsets_knn(X, y, feature_names, cv=5):
    """
    Compare KNN performance with different feature subsets.
    """
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.model_selection import cross_val_score, StratifiedKFold
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    
    print("\n" + "="*70)
    print("KNN PERFORMANCE WITH DIFFERENT FEATURE SUBSETS")
    print("="*70)
    
    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    
    results = []
    
    for k in range(1, len(feature_names) + 1):
        subset = feature_names[:k]
        
        if isinstance(X, pd.DataFrame):
            X_subset = X[subset]
        else:
            indices = [feature_names.index(f) for f in subset]
            X_subset = X[:, indices]
        
        # Create pipeline with scaling
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('knn', KNeighborsClassifier(n_neighbors=5))
        ])
        
        scores = cross_val_score(pipeline, X_subset, y, cv=cv_splitter, scoring='f1')
        
        results.append({
            'n_features': k,
            'features': subset[-1] if k > 0 else '',
            'mean_f1': scores.mean(),
            'std_f1': scores.std(),
            'improvement': 0
        })
        
        print(f"{k} features: {scores.mean():.4f} ± {scores.std():.4f}  | Added: {subset[-1]}")
    
    # Calculate improvement
    for i in range(1, len(results)):
        results[i]['improvement'] = results[i]['mean_f1'] - results[i-1]['mean_f1']
    
    results_df = pd.DataFrame(results)
    
    print("\n" + "-"*70)
    print("FEATURE IMPACT:")
    print("-"*70)
    for i, row in results_df.iterrows():
        if i == 0:
            continue
        impact = "📈 POSITIVE" if row['improvement'] > 0.01 else "📉 NEGATIVE" if row['improvement'] < -0.01 else "➡️  NEUTRAL"
        print(f"{impact} | Feature {row['n_features']}: {row['features']:30s} | Δ = {row['improvement']:+.4f}")
    
    return results_df


# Usage example:
"""
# Run diagnostics
diagnostics = diagnose_feature_issues(
    X=X_train,
    feature_names=ordered_features,
    y=y_train
)

# Compare subsets specifically for KNN
knn_results = compare_feature_subsets_knn(
    X=X_train,
    y=y_train,
    feature_names=ordered_features
)
"""

In [ ]:
diagnostics = diagnose_feature_issues(
    X=X_train,
    feature_names=ordered_features,
    y=y_train
)

In [ ]:
knn_results = compare_feature_subsets_knn(
    X=X_train,
    y=y_train,
    feature_names=ordered_features
)

## Algorithm 2 - Use TPE and TPD

### Feature Selection

In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING AND FEATURE SELECTION
# ============================================================================
selected_features = ['Total_power_efficiency', 'Total_power_delay']

[X_train, X_test, y_train, y_test, X_train_healthy] = preprocess_data(df, selected_features)
X_train.head()

In [ ]:
# ============================================================================
# STEP 2: IMPUTE + FEATURE SCALING
# ============================================================================

from sklearn.impute import SimpleImputer

def scale_features(X_train, X_test, X_train_healthy):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    X_train_healthy_imp = imputer.transform(X_train_healthy)

    # 2) Standardization (fit only on imputed training set)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_train_healthy_scaled = scaler.transform(X_train_healthy_imp)

    return X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer

[X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer] = scale_features(X_train, X_test, X_train_healthy)

### Model Training and Cross Validation

In [ ]:
trained_models_2_feat = train_all_models(X_train_scaled, y_train, X_train_healthy_scaled, contamination=0.05)

cv_summary_2_feat = cross_validate_all_models_with_metrics(
    trained_models,
    X_train_scaled,
    y_train,
    k=5,
    out_csv="cv_results_summary_2_feat.csv"
)

### Figure plotting of Cross Validation

In [ ]:
cv_summary_2_feat = crossvalidated_metrics_table(
    trained_models_2_feat, X_train_scaled, y_train
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Optional: to make plots look nicer
plt.style.use('seaborn-v0_8')  # or comment out if you prefer default

# --- Prepare data ---
models = cv_summary_2_feat['Model'].values
f1_mean = cv_summary_2_feat['F1_mean'].values
f1_std  = cv_summary_2_feat['F1_std'].values
types   = cv_summary_2_feat['Type'].values  # 'supervised', 'supervised_smote', 'anomaly'

# Assign a color per type
type_to_color = {
    'supervised': '#1f77b4',        # blue
    'supervised_smote': '#2ca02c',  # green
    'anomaly': '#d62728'            # red
}
colors = [type_to_color[t] for t in types]

# --- Create bar chart ---
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(models))

bars = ax.bar(x, f1_mean, yerr=f1_std, capsize=5, color=colors, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.set_ylabel('F1-score (mean ± std)')
ax.set_title('Cross-validated F1-score per Model')

# Build custom legend
handles = []
labels  = []
for t, c in type_to_color.items():
    handles.append(plt.Rectangle((0, 0), 1, 1, color=c))
    labels.append(t)
ax.legend(handles, labels, title='Model Type')

ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone

# Define the two best models to visualize
best_models = ["Decision Tree (Weighted)", "Random Forest (Weighted)", "XGBoost (Weighted)"]

# Prepare data
X = np.asarray(X_train_scaled)
y = np.asarray(y_train)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Create subplots: 2 models × 2 views (raw + normalized)
fig, axes = plt.subplots(len(best_models), 2, figsize=(12, 8))

for idx, model_name in enumerate(best_models):
    base_model = trained_models[model_name]['model']
    mtype = trained_models[model_name]['type']

    # Build estimator
    if mtype == 'supervised_smote':
        estimator = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', clone(base_model))
        ])
    elif mtype == 'supervised':
        estimator = clone(base_model)
    else:
        raise ValueError(f"{model_name} is not a supervised model.")

    # Cross-validated predictions
    y_pred_cv = cross_val_predict(estimator, X, y, cv=cv)

    # Confusion matrix
    cm = confusion_matrix(y, y_pred_cv)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    # Plot raw counts
    ax_raw = axes[idx, 0]
    im_raw = ax_raw.imshow(cm, cmap='coolwarm')
    ax_raw.set_title(f'{model_name} (Counts)')
    ax_raw.set_xticks([0, 1]); ax_raw.set_yticks([0, 1])
    ax_raw.set_xticklabels(['Pred Healthy', 'Pred Leakage'])
    ax_raw.set_yticklabels(['True Healthy', 'True Leakage'])
    for i in range(2):
        for j in range(2):
            text_color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax_raw.text(j, i, cm[i, j], ha='center', va='center', color=text_color)
    plt.colorbar(im_raw, ax=ax_raw, fraction=0.046, pad=0.04)

    # Plot normalized
    ax_norm = axes[idx, 1]
    im_norm = ax_norm.imshow(cm_norm, cmap='coolwarm', vmin=0, vmax=1)
    ax_norm.set_title(f'{model_name} (Normalized)')
    ax_norm.set_xticks([0, 1]); ax_norm.set_yticks([0, 1])
    ax_norm.set_xticklabels(['Pred Healthy', 'Pred Leakage'])
    ax_norm.set_yticklabels(['True Healthy', 'True Leakage'])
    for i in range(2):
        for j in range(2):
            text_color = 'white' if cm_norm[i, j] > 0.5 else 'black'
            ax_norm.text(j, i, f"{cm_norm[i, j]:.2f}", ha='center', va='center', color=text_color)
    plt.colorbar(im_norm, ax=ax_norm, fraction=0.046, pad=0.04)

# Final layout
plt.suptitle('Confusion Matrices for Top Models', y=1.02)
plt.tight_layout()
for ax in axes.flatten():
    ax.grid(False)
plt.show()

### Precision Recall curves (Sensitive to imbalance data)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.base import clone

# -------------------------------------------------------
# 1. Choose models to plot
# -------------------------------------------------------
models_to_plot = [
    "Random Forest (Weighted)",
    "Decision Tree (SMOTE)",
    "Logistic Regression (Weighted)",
    "XGBoost (SMOTE)",
]

X = np.asarray(X_train_scaled)
y = np.asarray(y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

plt.figure(figsize=(7, 6))

for name in models_to_plot:
    entry = trained_models[name]
    base_model = entry['model']
    mtype      = entry['type']

    # --- build estimator depending on type ---
    if mtype == 'supervised':
        estimator = clone(base_model)

    elif mtype == 'supervised_smote':
        estimator = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', clone(base_model))
        ])

    else:
        print(f"Skipping {name} (type '{mtype}' not supported for PR curve).")
        continue

    # --- cross-validated scores (out-of-fold) ---
    y_scores = cross_val_predict(
        estimator, X, y,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    precision, recall, _ = precision_recall_curve(y, y_scores)
    ap = average_precision_score(y, y_scores)

    plt.plot(recall, precision, label=f"{name} (AP = {ap:.3f})")

# baseline: proportion of positives (random classifier)
pos_ratio = y.mean()
plt.hlines(pos_ratio, 0, 1, colors='gray', linestyles='--',
           label=f"Baseline (pos ratio = {pos_ratio:.2f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Cross-validated Precision–Recall Curves")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


### ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import cross_val_predict

plt.figure(figsize=(7, 6))

models_to_plot = [
    "Random Forest (Weighted)",
    "Decision Tree (SMOTE)",
    "Logistic Regression (Weighted)",
    "XGBoost (SMOTE)",
]

X = np.asarray(X_train_scaled)
y = np.asarray(y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name in models_to_plot:
    entry = trained_models[name]
    base_model = entry['model']
    mtype      = entry['type']

    # Build estimator
    if mtype == 'supervised':
        estimator = clone(base_model)
    elif mtype == 'supervised_smote':
        estimator = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', clone(base_model))
        ])
    else:
        print(f"Skipping anomaly model: {name}")
        continue

    # CV predicted probabilities
    y_scores = cross_val_predict(
        estimator, X, y,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    fpr, tpr, _ = roc_curve(y, y_scores)
    auc = roc_auc_score(y, y_scores)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", lw=2)

# Random baseline
plt.plot([0, 1], [0, 1], 'k--', label="Random classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Cross-validated ROC Curves for Multiple Models")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


### Results - Threshold Value for the top 2 models

In [ ]:
from sklearn.tree import export_text

tree = trained_models_2_feat['Decision Tree (SMOTE)']['model']
print(export_text(tree, feature_names=['TPE','TPD']))

original_threshold = scaler.inverse_transform([[-0.42, 0]])  # 0 is a placeholder for TPD
print(original_threshold[0][0])  # Extract the TPE value


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# Get the trained decision tree model
tree = trained_models_2_feat['Decision Tree (Weighted)']['model']

# Plot the tree
plt.figure(figsize=(16, 10))  # Adjust size as needed
plot_tree(
    tree,
    feature_names=['TPE','TPD'],
    class_names=['Healthy', 'Leakage'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree Visualization")
plt.show()

### Comparison Table of F1 Score, Precision, Recall, and ROC value

In [ ]:
cv_results_2_feat = crossvalidated_metrics_table(
    trained_models_2_feat, X_train_scaled, y_train
)

cv_results_2_feat


### Tuning of 2 Features

In [ ]:
# Run XGboost tuning
best_xgb_est, best_xgb_params, best_xgb_score = tune_xgboost_weighted(
    X_train_scaled, y_train
)

In [ ]:
# Run Random Forest tuning
best_rf_est, best_rf_params, best_rf_score = tune_random_forest_weighted(
    X_train_scaled, y_train
)

In [ ]:
# Replace only the model with the best estimator
trained_models_2_feat['XGBoost (Weighted)']['model'] = best_xgb_est
trained_models_2_feat['Random Forest (Weighted)']['model'] = best_rf_est

# Evaluate
cv_results_2_feat_tuned = crossvalidated_metrics_table(
    trained_models_2_feat, X_train_scaled, y_train
)
cv_results_2_feat_tuned

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone

# Define the two best models to visualize
best_models = ["Decision Tree (Weighted)", "Random Forest (Weighted)", "XGBoost (Weighted)"]

# Prepare data
X = np.asarray(X_train_scaled)
y = np.asarray(y_train)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Create subplots: 2 models × 2 views (raw + normalized)
fig, axes = plt.subplots(len(best_models), 2, figsize=(12, 8))

for idx, model_name in enumerate(best_models):
    base_model = trained_models_2_feat[model_name]['model']
    mtype = trained_models_2_feat[model_name]['type']

    # Build estimator
    if mtype == 'supervised_smote':
        estimator = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', clone(base_model))
        ])
    elif mtype == 'supervised':
        estimator = clone(base_model)
    else:
        raise ValueError(f"{model_name} is not a supervised model.")

    # Cross-validated predictions
    y_pred_cv = cross_val_predict(estimator, X, y, cv=cv)

    # Confusion matrix
    cm = confusion_matrix(y, y_pred_cv)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    # Plot raw counts
    ax_raw = axes[idx, 0]
    im_raw = ax_raw.imshow(cm, cmap='coolwarm')
    ax_raw.set_title(f'{model_name} (Counts)')
    ax_raw.set_xticks([0, 1]); ax_raw.set_yticks([0, 1])
    ax_raw.set_xticklabels(['Pred Healthy', 'Pred Leakage'])
    ax_raw.set_yticklabels(['True Healthy', 'True Leakage'])
    for i in range(2):
        for j in range(2):
            text_color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax_raw.text(j, i, cm[i, j], ha='center', va='center', color=text_color)
    plt.colorbar(im_raw, ax=ax_raw, fraction=0.046, pad=0.04)

    # Plot normalized
    ax_norm = axes[idx, 1]
    im_norm = ax_norm.imshow(cm_norm, cmap='coolwarm', vmin=0, vmax=1)
    ax_norm.set_title(f'{model_name} (Normalized)')
    ax_norm.set_xticks([0, 1]); ax_norm.set_yticks([0, 1])
    ax_norm.set_xticklabels(['Pred Healthy', 'Pred Leakage'])
    ax_norm.set_yticklabels(['True Healthy', 'True Leakage'])
    for i in range(2):
        for j in range(2):
            text_color = 'white' if cm_norm[i, j] > 0.5 else 'black'
            ax_norm.text(j, i, f"{cm_norm[i, j]:.2f}", ha='center', va='center', color=text_color)
    plt.colorbar(im_norm, ax=ax_norm, fraction=0.046, pad=0.04)

# Final layout
plt.suptitle('Confusion Matrices for Top Models', y=1.02)
plt.tight_layout()
for ax in axes.flatten():
    ax.grid(False)
plt.show()

# Trained Model Comparison

## Improvement using multiple features

In [ ]:
cv_results_1_feat['Features'] = '1'
cv_results_2_feat_tuned['Features'] = '2'

# Merge on Model
comparison = cv_results_1_feat.merge(
    cv_results_2_feat_tuned,
    on="Model",
    suffixes=("_1feat", "_2feat")
)

# Keep only relevant metrics
metrics = ["Precision", "Recall", "F1-Score", "ROC-AUC"]
for m in metrics:
    comparison[f"{m}_Improvement"] = (
        comparison[f"{m}_2feat"] - comparison[f"{m}_1feat"]
    )
    
for m in metrics:
    comparison[f"{m}_PctImprovement"] = (
        (comparison[f"{m}_2feat"] - comparison[f"{m}_1feat"]) /
        comparison[f"{m}_1feat"] * 100
    )
import matplotlib.pyplot as plt

# Example: F1-score comparison
comparison.plot(
    x="Model",
    y=["F1-Score_1feat", "F1-Score_2feat"],
    kind="bar",
    figsize=(10,6)
)
plt.title("F1-Score Comparison (1 vs 2 Features)")
plt.ylabel("F1-Score")
plt.xticks(rotation=45, ha="right")
plt.show()

# Example: Improvement only
comparison.plot(
    x="Model",
    y="F1-Score_Improvement",
    kind="bar",
    color="skyblue",
    figsize=(10,6)
)
plt.title("Improvement in F1-Score (2 Features vs 1 Feature)")
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("Δ F1-Score")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
metrics = ["Precision", "Recall", "F1-Score", "ROC-AUC"]

# Loop through each metric and plot percentage improvement
for m in metrics:
    plt.figure(figsize=(10,6))
    plt.bar(comparison["Model"], comparison[f"{m}_PctImprovement"], color="skyblue")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.title(f"Percentage Improvement in {m} (2 Features vs 1 Feature)")
    plt.ylabel("% Improvement")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

# Unsupervised (Anomaly) Model Tuning

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

# ---------------------------------------------------------------------
# 1) Define base anomaly models
# ---------------------------------------------------------------------
def get_anomaly_models():
    return {
        'Isolation Forest': IsolationForest(
            contamination=0.05,
            random_state=42,
            n_estimators=100,
            n_jobs=-1
        ),
        'One-Class SVM': OneClassSVM(
            nu=0.05,
            kernel='rbf',
            gamma='scale'
        ),
        'Local Outlier Factor': LocalOutlierFactor(
            contamination=0.05,
            novelty=True,     # IMPORTANT for using on test data
            n_neighbors=25
        )
    }

# ---------------------------------------------------------------------
# 2) Simple parameter grids for each anomaly model
#    (adjust ranges based on runtime/data size)
# ---------------------------------------------------------------------
ANOMALY_PARAM_GRIDS = {
    'Isolation Forest': {
        'contamination': [0.01, 0.03, 0.05, 0.1],
        'n_estimators': [100, 200],
        'max_samples': ['auto', 0.7, 0.9]
    },
    'One-Class SVM': {
        'nu': [0.01, 0.05, 0.1],
        'gamma': ['scale', 0.1, 1.0]
    },
    'Local Outlier Factor': {
        'contamination': [0.01, 0.03, 0.05, 0.1],
        'n_neighbors': [10, 20, 30]
    }
}

# ---------------------------------------------------------------------
# 3) Helper: compute anomaly scores (higher = more normal)
# ---------------------------------------------------------------------
def get_anomaly_scores(model, X):
    """
    Returns scores where larger values mean 'more normal'.
    Works for IsolationForest, OneClassSVM, LOF (novelty=True).
    """
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
    elif hasattr(model, "score_samples"):
        scores = model.score_samples(X)
    else:
        raise ValueError("Model has neither decision_function nor score_samples.")
    return scores  # we assume anomalies are in the lower tail of scores

# ---------------------------------------------------------------------
# 4) Helper: find the best threshold on scores for a given fold
# ---------------------------------------------------------------------
def find_best_threshold(y_true, scores, n_grid=50):
    """
    y_true: {0,1}, with 1 = leakage (anomaly)
    scores: larger = more normal
    Returns (best_threshold, best_f1).
    """
    # scan thresholds between low and high quantiles of scores
    qs = np.linspace(0.01, 0.99, n_grid)
    cand_thresholds = np.quantile(scores, qs)

    best_t, best_f1 = None, -np.inf
    for t in cand_thresholds:
        # anomaly if score < t
        y_pred = (scores < t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t

    return best_t, best_f1

# ---------------------------------------------------------------------
# 5) Main tuner: semi-supervised tuning for anomaly models
# ---------------------------------------------------------------------
def tune_anomaly_models(X_scaled, y, k=5):
    """
    Semi-supervised tuning of anomaly detection models:
    - train only on healthy in each fold
    - choose params + threshold maximizing F1
    """
    X = np.asarray(X_scaled)
    y = np.asarray(y)

    models = get_anomaly_models()
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    records = []
    best_models = {}

    for name, base_model in models.items():
        print(f"\n=== Tuning {name} ===")
        param_grid = ANOMALY_PARAM_GRIDS[name]

        best_global_f1 = -np.inf
        best_global_params = None
        best_global_threshold = None

        # loop over hyperparameter combinations
        for params in ParameterGrid(param_grid):
            f1_folds = []
            thr_folds = []

            for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
                X_tr, X_val = X[train_idx], X[val_idx]
                y_tr, y_val = y[train_idx], y[val_idx]

                # train only on healthy data
                X_tr_healthy = X_tr[y_tr == 0]
                if X_tr_healthy.shape[0] == 0:
                    continue  # safety

                model = clone(base_model).set_params(**params)
                model.fit(X_tr_healthy)

                # scores on validation (healthy + leakage)
                scores_val = get_anomaly_scores(model, X_val)

                # tune threshold on this fold
                thr, f1 = find_best_threshold(y_val, scores_val)
                f1_folds.append(f1)
                thr_folds.append(thr)

            if len(f1_folds) == 0:
                continue

            mean_f1 = float(np.mean(f1_folds))
            std_f1 = float(np.std(f1_folds))

            records.append({
                'Model': name,
                'Params': params,
                'F1_mean': mean_f1,
                'F1_std': std_f1,
                'Thr_mean': float(np.mean(thr_folds))
            })

            print(f"  params={params} | F1_mean={mean_f1:.3f} ± {std_f1:.3f}")

            # update global best
            if mean_f1 > best_global_f1:
                best_global_f1 = mean_f1
                best_global_params = params
                best_global_threshold = float(np.mean(thr_folds))

        # refit best model on ALL healthy data
        if best_global_params is not None:
            print(f"→ Best for {name}: F1={best_global_f1:.3f}, params={best_global_params}, "
                  f"thr={best_global_threshold:.4f}")

            X_all_healthy = X[y == 0]
            best_model = clone(base_model).set_params(**best_global_params)
            best_model.fit(X_all_healthy)

            best_models[name] = {
                'model': best_model,
                'threshold': best_global_threshold,
                'best_params': best_global_params,
                'F1_cv': best_global_f1
            }

    df_results = pd.DataFrame(records)
    return df_results, best_models


In [ ]:
df_anom_tuning, tuned_anomaly_models = tune_anomaly_models(
    X_scaled=X_train_scaled,
    y=y_train,
    k=5
)

print(df_anom_tuning.sort_values('F1_mean', ascending=False).head())

## To predict on test data

In [ ]:
best_if = tuned_anomaly_models['Isolation Forest']
model_if = best_if['model']
thr_if   = best_if['threshold']

scores_test = get_anomaly_scores(model_if, X_test_scaled)
y_pred_if   = (scores_test < thr_if).astype(int)  # 1 = leakage, 0 = healthy

In [ ]:
# Build tuned anomaly dict in the same format as trained_models_2_feat
tuned_anomaly_dict = {
    name: {
        "model": entry["model"],
        "type": "anomaly",
        "threshold": entry["threshold"],   # <<< KEEP THIS
        "best_params": entry.get("best_params", None),
        "F1_cv_tuner": entry.get("F1_cv", None)
    }
    for name, entry in tuned_anomaly_models.items()
}


# Merge with your supervised + baseline models
combined_models = trained_models_2_feat.copy()
combined_models.update(tuned_anomaly_dict)

# Evaluate
cv_results_2_feat_tuned = crossvalidated_metrics_table(
    combined_models, X_train_scaled, y_train
)
cv_results_2_feat_tuned

## Comparison after tuning the Unsupervised

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ================================
# Add feature labels
# ================================
cv_results_1_feat['Features'] = '1'
cv_results_2_feat['Features'] = '2'
cv_results_2_feat_tuned['Features'] = '3'

# ================================
# Merge all three tables on Model
# ================================
comparison = (
    cv_results_1_feat.merge(cv_results_2_feat, on="Model", suffixes=("_1", "_2"))
                   .merge(cv_results_2_feat_tuned, on="Model")
)

# Rename tuned columns to keep consistent suffixes
comparison = comparison.rename(columns={
    col: col + "_3" for col in comparison.columns 
    if col not in ["Model"] and not col.endswith("_1") and not col.endswith("_2")
})

metrics = ["Precision", "Recall", "F1-Score", "ROC-AUC"]

# ================================
# Compute improvements
# ================================
for m in metrics:
    # Absolute improvements
    comparison[f"{m}_2vs1"] = comparison[f"{m}_2"] - comparison[f"{m}_1"]
    comparison[f"{m}_3vs2"] = comparison[f"{m}_3"] - comparison[f"{m}_2"]
    comparison[f"{m}_3vs1"] = comparison[f"{m}_3"] - comparison[f"{m}_1"]

    # Percentage improvements
    comparison[f"{m}_Pct_2vs1"] = (comparison[f"{m}_2vs1"] / comparison[f"{m}_1"]) * 100
    comparison[f"{m}_Pct_3vs2"] = (comparison[f"{m}_3vs2"] / comparison[f"{m}_2"]) * 100
    comparison[f"{m}_Pct_3vs1"] = (comparison[f"{m}_3vs1"] / comparison[f"{m}_1"]) * 100

# ================================
# Plot: F1-score comparison 1 vs 2 vs 3
# ================================
comparison.set_index("Model")[["F1-Score_1", "F1-Score_2", "F1-Score_3"]].plot(
    kind="bar", figsize=(12, 6)
)
plt.title("F1-Score Comparison (1 Feature vs 2 Features vs Tuned)")
plt.ylabel("F1-Score")
plt.xticks(rotation=45, ha="right")
plt.show()

# ================================
# Plot: Improvements (example: 3 vs 2)
# ================================
comparison.plot(
    x="Model",
    y="F1-Score_3vs2",
    kind="bar",
    color="skyblue",
    figsize=(10, 6)
)
plt.title("Improvement in F1-Score (Tuned vs 2 Features)")
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("Δ F1-Score")
plt.xticks(rotation=45, ha="right")
plt.show()
